# 普通模型 vs 鲁棒模型：对抗鲁棒性对比

从 `resnet-18-apgd.ipynb` 拆分出的**独立**子任务，做「普通(基础训练) 模型 vs 鲁棒(PGD-AT) 模型」在各攻击下的 **Robust Acc** 对比。与主 notebook 的「诚实评估」同款分组柱状图，并在其上**多加 DeepFool 一档**（FGSM / PGD-20 / APGD / DeepFool 画在同一张图里）。

**前置条件**：两份已训练好的权重
- 基础模型：`resnet18_cifar10_best.pth`
- 鲁棒模型：`resnet18_cifar10_pgd_at.pth`

**约定（与主项目一致）**：攻击在 `[0,1]` 像素空间进行，归一化作为模型第一层（`atk_model`），
CIFAR 版 ResNet-18 stem 改 `conv1=3x3 stride1` + `maxpool=Identity`。

依次运行两个单元：① 环境与双模型加载 → ② FGSM / PGD-20 / APGD / DeepFool 的 Clean/Robust Acc 对比表 + 同一张分组柱状图。

> 注：DeepFool 求最小扰动、无 ε 预算，迭代到刚好越界 → 两模型 Robust Acc 都≈0，仅作为与三种 ε 攻击并排的对照档。


In [ ]:
# ==================== ① 环境、双模型加载、评估 helper（自包含）====================
# 仅保留「普通 vs 鲁棒」对比所需的最小集合：从主 notebook 的「优化版本」单元裁剪
# 而来（去掉 t-SNE / 特征提取 / 单攻击可视化），并并入鲁棒模型的「加载」逻辑。
import os
import math
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import torchattacks
from tqdm import tqdm
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, datasets

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

plt.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.size': 11, 'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'axes.labelsize': 11, 'legend.fontsize': 10,
    'axes.grid': True, 'grid.linestyle': '--', 'grid.alpha': 0.3, 'axes.axisbelow': True,
    'axes.spines.top': False, 'axes.spines.right': False,
    'legend.frameon': True, 'legend.framealpha': 0.9,
})

# ---------- CIFAR 版 ResNet-18（stem 改造：3x3 stride1 + maxpool=Identity）----------
def create_resnet18(num_classes=10):
    from torchvision.models import resnet18
    m = resnet18(weights=None)
    m.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

# ---------- 归一化包装层：放进模型，攻击在 [0,1] 空间进行 ----------
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD  = [0.229, 0.224, 0.225]

class Normalize(nn.Module):
    def __init__(self, mean, std):
        super().__init__()
        self.register_buffer('mean', torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer('std',  torch.tensor(std).view(1, 3, 1, 1))

    def forward(self, x):
        return (x - self.mean) / self.std

def _find_ckpt(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"未找到权重，请把路径加入候选列表：{candidates}")

# 基础（干净训练）模型 → atk_model（普通模型）
BASE_CKPT = _find_ckpt([
    '/kaggle/input/notebooks/liangliguo/cifar/resnet18_cifar10_best.pth',
    'to/dest/resnet18_cifar10_best.pth',
    'resnet18_cifar10_best.pth',
])
model = create_resnet18().to(device)
model.load_state_dict(torch.load(BASE_CKPT, map_location=device)['model_state_dict'])
model.eval()
atk_model = nn.Sequential(Normalize(NORM_MEAN, NORM_STD), model).to(device).eval()
print(f"✓ 普通模型已加载: {BASE_CKPT}")

# 鲁棒（PGD-AT）模型 → atk_model_robust（鲁棒模型）
ROBUST_CKPT = _find_ckpt([
    '/kaggle/input/notebooks/liangliguo/cifar/resnet18_cifar10_pgd_at.pth',
    'resnet18_cifar10_pgd_at.pth',
    'to/dest/resnet18_cifar10_pgd_at.pth',
])
robust_net = create_resnet18().to(device)
robust_net.load_state_dict(torch.load(ROBUST_CKPT, map_location=device)['model_state_dict'])
robust_net.eval()
atk_model_robust = nn.Sequential(Normalize(NORM_MEAN, NORM_STD), robust_net).to(device).eval()
print(f"✓ 鲁棒模型已加载: {ROBUST_CKPT}")

# ---------- 测试集（保持 [0,1]，归一化交给 atk_model 内部）----------
DATA_ROOT = next((p for p in [
    '/kaggle/input/datasets/pankrzysiu/cifar10-python',   # Kaggle: pankrzysiu/cifar10-python
    './data',                                             # 本地回退
] if os.path.isdir(os.path.join(p, 'cifar-10-batches-py'))), './data')
test_transform = transforms.Compose([transforms.ToTensor()])
_need_dl = not os.path.isdir(os.path.join(DATA_ROOT, 'cifar-10-batches-py'))
test_dataset = datasets.CIFAR10(root=DATA_ROOT, train=False, download=_need_dl, transform=test_transform)
test_loader = DataLoader(Subset(test_dataset, list(range(10000))), batch_size=32, shuffle=False, num_workers=2)
print(f"✓ 测试集就绪（{len(test_loader.dataset)} 张, [0,1] 空间）")

# ---------- FGSM（[0,1] 空间，结尾 clamp，口径与 torchattacks 一致）----------
class CustomFGSM:
    def __init__(self, model, eps=8 / 255):
        self.model = model
        self.eps = eps

    def __call__(self, images, labels):
        images = images.clone().detach().requires_grad_(True)
        loss = torch.nn.functional.cross_entropy(self.model(images), labels)
        grad = torch.autograd.grad(loss, images)[0]
        return torch.clamp(images + self.eps * grad.sign(), 0, 1).detach()

# ---------- 生成对抗样本（显存友好；返回 adv/orig/labels/L2）----------
def generate_adversarial_optimized(attack, name, fwd_model, dataloader, device, max_samples=1000):
    fwd_model.eval()
    advs, origs, lbls_l, l2s, collected = [], [], [], [], 0
    total_batches = math.ceil(max_samples / dataloader.batch_size)
    for imgs, lbls in tqdm(dataloader, total=total_batches, desc=f"Generating {name} (n={max_samples})"):
        if collected >= max_samples:
            break
        imgs, lbls = imgs.to(device), lbls.to(device)
        adv = attack(imgs, lbls).detach()
        with torch.no_grad():
            l2 = torch.norm((adv - imgs).view(imgs.size(0), -1), dim=1).cpu()
        advs.append(adv.cpu()); origs.append(imgs.detach().cpu()); lbls_l.append(lbls.cpu()); l2s.append(l2)
        collected += imgs.size(0)
        del imgs, lbls, adv
    return (torch.cat(advs)[:max_samples], torch.cat(origs)[:max_samples],
            torch.cat(lbls_l)[:max_samples], torch.cat(l2s)[:max_samples].numpy())

# ---------- 白盒攻击评估：条件攻击成功率 + Clean/Robust Acc + L2 + ρ_adv ----------
def evaluate_attack_optimized(attack, name, fwd_model, dataloader, device, num_batches=10):
    """返回 (asr, clean_acc, robust_acc, avg_l2, avg_rho)。
      asr        = 条件攻击成功率 = #(干净分对 且 对抗分错) / #(干净分对)
      robust_acc = #(对抗分对) / 总数；avg_rho = 平均相对扰动 ‖r‖₂/‖x‖₂
    注：robust_acc 与 asr 独立，二者相加不一定 = 100%。"""
    fwd_model.eval()
    total = clean_correct = robust_correct = flipped = 0
    total_l2 = total_rho = 0.0
    eval_batches = min(num_batches, len(dataloader))
    for i, (imgs, lbls) in enumerate(tqdm(dataloader, total=eval_batches,
                                          desc=f"Evaluating {name} (≈{eval_batches * dataloader.batch_size} 张)")):
        if i >= num_batches:
            break
        imgs, lbls = imgs.to(device), lbls.to(device)
        adv = attack(imgs, lbls).detach()
        with torch.no_grad():
            clean_ok = (fwd_model(imgs).argmax(1) == lbls)
            adv_ok = (fwd_model(adv).argmax(1) == lbls)
            clean_correct += clean_ok.sum().item()
            robust_correct += adv_ok.sum().item()
            flipped += (clean_ok & ~adv_ok).sum().item()
            total += lbls.size(0)
            r = torch.norm((adv - imgs).view(imgs.size(0), -1), dim=1)
            total_l2 += r.sum().item()
            total_rho += (r / torch.norm(imgs.view(imgs.size(0), -1), dim=1).clamp_min(1e-12)).sum().item()
        del imgs, lbls, adv
    return (flipped / max(1, clean_correct), clean_correct / total,
            robust_correct / total, total_l2 / total, total_rho / total)

print("\n✓ helper 就绪：CustomFGSM / generate_adversarial_optimized / evaluate_attack_optimized")

In [ ]:
# ==================== ② 普通 vs 鲁棒：Robust Acc 对比（FGSM/PGD-20/APGD/DeepFool）====================
# 说明：
#   - 攻击由弱到强：FGSM(单步) < PGD-20(标准基准) < APGD(自适应)；DeepFool 求最小扰动、
#     无 ε 预算，迭代到刚好越界，几乎必然成功，故两模型 Robust Acc 都≈0（与三种 ε 攻击并排作对照）。
#   - 每种攻击都按「白盒」对被评估模型单独构造；统一用 Clean/Robust Acc + 条件攻击成功率口径。
#   - DeepFool 较慢，单独限制评估样本数（DF_BATCHES≈1000 张），其余攻击用全量 10000。
#   - 把四种攻击下两模型的 Robust Acc 画成同一张分组柱状图（与主 notebook 诚实评估图同款，多加 DeepFool 一档），存 SVG/PDF。
import numpy as np
import matplotlib.pyplot as plt
import torchattacks

def _build_fgsm(fm):
    return CustomFGSM(fm, eps=8 / 255)                                  # 单步弱攻击

def _build_pgd20(fm):
    return torchattacks.PGD(fm, eps=8/255, alpha=2/255, steps=20, random_start=True)

def _build_apgd(fm):
    return torchattacks.APGD(fm, norm='Linf', eps=8/255, steps=20, n_restarts=1, loss='ce')

def _build_deepfool(fm):
    return torchattacks.DeepFool(fm, steps=50, overshoot=0.02)          # 最小 L2 扰动, 无 ε 预算

EVAL_BATCHES = len(test_loader)   # 固定 ε 攻击：全量 10000
DF_BATCHES   = 32                 # DeepFool 慢：约 1000 张

# (显示名, builder, 短名, 评估 batch 数)
ATTACKS = [("FGSM   (单步, 弱)",         _build_fgsm,     "FGSM",     EVAL_BATCHES),
           ("PGD-20 (Linf 8/255)",      _build_pgd20,    "PGD-20",   EVAL_BATCHES),
           ("APGD   (Linf 8/255, 强)",  _build_apgd,     "APGD",     EVAL_BATCHES),
           ("DeepFool (最小扰动, 无ε)", _build_deepfool, "DeepFool", DF_BATCHES)]
MODELS = [("普通模型", atk_model), ("鲁棒模型", atk_model_robust)]

# results[short][模型tag] = (clean_acc, robust_acc, asr)
results = {short: {} for _, _, short, _ in ATTACKS}

print("=" * 66)
print("普通模型 vs 鲁棒模型：Clean / Robust Acc 对比")
for atk_name, builder, short, n_batches in ATTACKS:
    print("-" * 66)
    print(f"[{atk_name}]")
    print(f"{'模型':<8}{'Clean Acc':>12}{'Robust Acc':>12}{'攻击成功率':>12}")
    for tag, fm in MODELS:
        atk = builder(fm)
        asr, clean_acc, rob_acc, _, _ = evaluate_attack_optimized(
            atk, atk_name, fm, test_loader, device, num_batches=n_batches)
        print(f"{tag:<8}{clean_acc:>11.2%}{rob_acc:>12.2%}{asr:>12.2%}")
        results[short][tag] = (clean_acc, rob_acc, asr)
print("-" * 66)
print("解读：")
print("- 对抗训练若有效：鲁棒模型在各攻击下 Robust Acc 都应高于普通模型。")
print("- 固定 ε 阶梯 FGSM < PGD-20 < APGD：同一模型 Robust Acc 应依次下降。")
print("- DeepFool 无 ε 预算、迭代到刚好越界 → 几乎必然成功，故两模型 Robust Acc 都≈0。")
print("- 攻击成功率为条件口径，故 Robust Acc + 攻击成功率 不一定 = 100%。")
print("=" * 66)

# ---------- 分组柱状图：四种攻击下两模型的 Robust Acc（FGSM/PGD-20/APGD + DeepFool）----------
attack_order = [short for _, _, short, _ in ATTACKS]
x = np.arange(len(attack_order))
width = 0.36
plain_rob = [results[s]["普通模型"][1] * 100 for s in attack_order]
robust_rob = [results[s]["鲁棒模型"][1] * 100 for s in attack_order]
plain_clean = results[attack_order[0]]["普通模型"][0] * 100   # clean 与攻击无关，取其一
robust_clean = results[attack_order[0]]["鲁棒模型"][0] * 100

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - width / 2, plain_rob, width, label="Standard model", color="#8C8C8C", edgecolor="white")
b2 = ax.bar(x + width / 2, robust_rob, width, label="Robust model (PGD-AT)", color="#C44E52", edgecolor="white")
ax.axhline(plain_clean, color="#8C8C8C", ls="--", lw=1, alpha=0.8)
ax.axhline(robust_clean, color="#C44E52", ls="--", lw=1, alpha=0.8)
ax.text(len(attack_order) - 0.5, plain_clean + 1, f"Standard clean {plain_clean:.0f}%",
        color="#5A5A5A", fontsize=8, ha="right")
ax.text(len(attack_order) - 0.5, robust_clean + 1, f"Robust clean {robust_clean:.0f}%",
        color="#C44E52", fontsize=8, ha="right")
for bars in (b1, b2):
    for r in bars:
        ax.annotate(f"{r.get_height():.1f}", (r.get_x() + r.get_width() / 2, r.get_height()),
                    textcoords="offset points", xytext=(0, 3), ha="center", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(attack_order)
ax.set_ylabel("Robust Accuracy (%)")
ax.set_ylim(0, 100)
ax.set_title("Robust Accuracy: Standard vs Robust (FGSM / PGD-20 / APGD / DeepFool)", pad=12)
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig("robustness_comparison.svg")
fig.savefig("robustness_comparison.pdf")
print("✓ 鲁棒性对比矢量图已保存：robustness_comparison.svg / .pdf")
plt.show()
